In [7]:
# ==========================================
# 1️⃣ Import Libraries
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


# ==========================================
# 2️⃣ Load Dataset (EDA)
# ==========================================
df = pd.read_csv('Mall_Customers.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:\n", df.head())
print("\nInfo:\n")
print(df.info())
print("\nMissing Values:\n", df.isnull().sum())


# ==========================================
# 3️⃣ One-Hot Encode Gender
# ==========================================
# Convert categorical Gender into numeric columns
df_encoded = pd.get_dummies(df, columns=['Genre'], drop_first=True)

# Now 'Genre_Male' column will be created
print("\nColumns after Encoding:\n", df_encoded.columns)


# ==========================================
# 4️⃣ Feature Selection
# ==========================================
# Drop CustomerID (not useful for clustering)
X = df_encoded.drop(columns=['CustomerID'])

print("\nFeatures used for clustering:\n", X.columns)


# ==========================================
# 5️⃣ Feature Scaling
# ==========================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ==========================================
# 6️⃣ Apply K-Means Clustering
# ==========================================
kmeans = KMeans(n_clusters=5, random_state=42)
df_encoded['Cluster'] = kmeans.fit_predict(X_scaled)

centroids = kmeans.cluster_centers_


# ==========================================
# 7️⃣ Interactive 3D Plot (Income, Spending, Age)
# ==========================================
# NOTE: Even though clustering used all features,
# we visualize only 3 major numeric ones.

fig = go.Figure()

# Customers
fig.add_trace(go.Scatter3d(
    x=X_scaled[:, X.columns.get_loc('Annual Income (k$)')],
    y=X_scaled[:, X.columns.get_loc('Spending Score (1-100)')],
    z=X_scaled[:, X.columns.get_loc('Age')],
    mode='markers',
    marker=dict(
        size=5,
        color=df_encoded['Cluster'],
        colorscale='Viridis',
        opacity=0.8
    ),
    text=df[['Annual Income (k$)',
             'Spending Score (1-100)',
             'Age',
             'Genre']],
    hovertemplate=
        'Income: %{text[0]}k$<br>' +
        'Spending Score: %{text[1]}<br>' +
        'Age: %{text[2]}<br>' +
        'Gender: %{text[3]}<extra></extra>',
    name='Customers'
))

# Centroids
fig.add_trace(go.Scatter3d(
    x=centroids[:, X.columns.get_loc('Annual Income (k$)')],
    y=centroids[:, X.columns.get_loc('Spending Score (1-100)')],
    z=centroids[:, X.columns.get_loc('Age')],
    mode='markers',
    marker=dict(
        size=12,
        color='red',
        symbol='x'
    ),
    name='Centroids'
))

fig.update_layout(
    title='Interactive 3D K-Means Clustering (All Features Used)',
    scene=dict(
        xaxis_title='Scaled Annual Income',
        yaxis_title='Scaled Spending Score',
        zaxis_title='Scaled Age'
    ),
    width=900,
    height=700
)

fig.show()


# ==========================================
# 8️⃣ Cluster Analysis
# ==========================================
cluster_analysis = df_encoded.groupby('Cluster').mean()

print("\nCluster Analysis:\n")
print(cluster_analysis)

Shape: (200, 5)

First 5 rows:
    CustomerID   Genre  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40

Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Genre                   200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income (k$)      200 non-null    int64 
 4   Spending Score (1-100)  200 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 7.9+ KB
None

Missing Va


Cluster Analysis:

         CustomerID        Age  Annual Income (k$)  Spending Score (1-100)  \
Cluster                                                                      
0         65.333333  56.470588           46.098039               39.313725   
1        159.500000  39.500000           85.150000               14.050000   
2        100.809524  28.690476           60.904762               70.238095   
3        151.510204  37.897959           82.122449               54.448980   
4         50.526316  27.315789           38.842105               56.210526   

         Genre_Male  
Cluster              
0          0.509804  
1          1.000000  
2          1.000000  
3          0.000000  
4          0.000000  
